# BinderDiffuser — Colab GPU run

Run the GPU-bound stages of BinderDiffuser end-to-end on a Colab T4/A100
and produce a real cohort of PD-L1 binder designs with measured metrics.

**Runtime:** Edit → Notebook settings → Hardware accelerator → **GPU** (T4 free, A100 if available).

**What this notebook does:**
1. Install RFdiffusion, ProteinMPNN, ColabFold, and BinderDiffuser.
2. Download PD-L1 (PDB 5O45) and trim to chain B (the IgV domain).
3. Generate 10 backbones with motif-scaffolded RFdiffusion (~5 min on T4).
4. Design 4 sequences per backbone with ProteinMPNN (~2 min on T4).
5. Re-fold every sequence with ColabFold/AF2 in multimer mode (~15 min on T4).
6. Compute scRMSD, scTM, mean pLDDT, ipTM, pAE_interface for each design.
7. Filter and rank, and emit the README scatter + violin plots.
8. Download the run artifacts as a tarball.

**Total runtime on T4:** ~25 minutes for 10 backbones × 4 sequences = 40 designs.

**Cost:** Free on T4, ~$0.50 on A100.

**Tip:** Bump `NUM_BACKBONES` and `SEQS_PER_BACKBONE` once you've confirmed
the small run works.

## 0. Sanity check the runtime

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import sys; print('python:', sys.version)

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07
python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


## 1. Install everything

RFdiffusion + ProteinMPNN + ColabFold + BinderDiffuser. First-run install takes
~10 minutes; subsequent runs are cached if you keep the Colab session warm.

In [2]:
%cd /content

# 1a. ColabFold (the AF2 frontend). Use the official setup script.
!if [ ! -f COLABFOLD_READY ]; then \
    pip install -q --no-warn-conflicts "colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold" && \
    pip install -q -U "jax[cuda12_pip]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html && \
    touch COLABFOLD_READY ; \
  fi

/content
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.4/248.4 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.3/374.3 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.0/101.0 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 67.2 MB/s eta 0:00:00


In [3]:
# 1b. RFdiffusion + its DGL/SE3-Transformer deps.
!if [ ! -d /content/RFdiffusion ]; then \
    git clone https://github.com/RosettaCommons/RFdiffusion.git /content/RFdiffusion && \
    pip install -q -e /content/RFdiffusion && \
    pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.3/cu121/repo.html && \
    pip install -q hydra-core pyrsistent ; \
  fi

# Download RFdiffusion model weights (~1.5 GB). Skip if already present.
!mkdir -p /content/RFdiffusion/models && cd /content/RFdiffusion/models && \
  if [ ! -f Base_ckpt.pt ]; then \
    wget -q http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt && \
    wget -q http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt ; \
  fi

Cloning into '/content/RFdiffusion'...
remote: Enumerating objects: 753, done.
remote: Counting objects: 100% (546/546), done.
remote: Compressing objects: 100% (340/340), done.
remote: Total 753 (delta 337), reused 236 (delta 202), pack-reused 207 (from 1)
Receiving objects: 100% (753/753), 10.18 MiB | 25.56 MiB/s, done.
Resolving deltas: 100% (385/385), done.
  Preparing metadata (setup.py) ... done
ERROR: Could not find a version that satisfies the requirement se3-transformer (from rfdiffusion) (from versions: none)
ERROR: No matching distribution found for se3-transformer


In [4]:
# 1c. ProteinMPNN.
!if [ ! -d /content/ProteinMPNN ]; then \
    git clone https://github.com/dauparas/ProteinMPNN.git /content/ProteinMPNN ; \
  fi

Cloning into '/content/ProteinMPNN'...
remote: Enumerating objects: 634, done.
remote: Counting objects: 100% (256/256), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 634 (delta 137), reused 132 (delta 132), pack-reused 378 (from 1)
Receiving objects: 100% (634/634), 119.90 MiB | 22.33 MiB/s, done.
Resolving deltas: 100% (290/290), done.


In [5]:
# 1d. BinderDiffuser itself.
!if [ ! -d /content/BinderDiffuser ]; then \
    git clone https://github.com/deepmind11/BinderDiffuser.git /content/BinderDiffuser && \
    pip install -q -e /content/BinderDiffuser ; \
  fi
%cd /content/BinderDiffuser

Cloning into '/content/BinderDiffuser'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 147 (delta 46), reused 146 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 810.17 KiB | 13.97 MiB/s, done.
Resolving deltas: 100% (46/46), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for binderdiffuser (pyproject.toml) ... done
/content/BinderDiffuser


## 2. Set up the PD-L1 target

Download PDB 5O45 from RCSB and trim to chain B (the PD-L1 IgV domain;
chains H and L are the antibody Fv we discard for binder design).

In [6]:
import os
from pathlib import Path

TARGET_DIR = Path('/content/BinderDiffuser/examples/pdl1_binder/target')
TARGET_DIR.mkdir(parents=True, exist_ok=True)

if not (TARGET_DIR / '5o45_full.pdb').exists():
    !wget -q https://files.rcsb.org/download/5O45.pdb -O {TARGET_DIR}/5o45_full.pdb

# Trim to chain B.
from Bio.PDB import PDBIO, PDBParser, Select

class ChainBOnly(Select):
    def accept_chain(self, c):
        return c.id == 'B'
    def accept_residue(self, r):
        return r.id[0] == ' '  # standard residues only

io = PDBIO()
io.set_structure(PDBParser(QUIET=True).get_structure('s', TARGET_DIR / '5o45_full.pdb'))
io.save(str(TARGET_DIR / 'pdl1_5o45.pdb'), ChainBOnly())
print('wrote', TARGET_DIR / 'pdl1_5o45.pdb')

wrote /content/BinderDiffuser/examples/pdl1_binder/target/pdl1_5o45.pdb


## 3. Configure the run

Small parameters here so it finishes quickly. Bump these after confirming
the pipeline works end-to-end.

In [7]:
NUM_BACKBONES = 10        # RFdiffusion designs
SEQS_PER_BACKBONE = 4     # ProteinMPNN sequences per backbone
BINDER_LEN_MIN = 60
BINDER_LEN_MAX = 80
MOTIF_RESIDUES = list(range(56, 66))  # PD-L1 56-65, the PD-1-binding loop
DIFFUSER_T = 50           # diffusion steps; lower = faster, lower quality
RUN_NAME = 'colab_smoke_test'
OUTPUT_DIR = Path('/content/BinderDiffuser/runs') / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('writing to', OUTPUT_DIR)

writing to /content/BinderDiffuser/runs/colab_smoke_test


## 4. Generate backbones with motif-scaffolded RFdiffusion

Each call to `run_inference.py` produces one backbone PDB. We loop
`NUM_BACKBONES` times with different seeds to get a diverse cohort.

On a T4, each backbone takes ~30 seconds. On an A100, ~10 seconds.

In [8]:
import random
from binderdiffuser.diffusion.motif_spec import MotifSpec, build_contig_string
from binderdiffuser.targets import resolve_target_motif

TARGET_PDB = TARGET_DIR / 'pdl1_5o45.pdb'
BACKBONES_DIR = OUTPUT_DIR / 'backbones'
BACKBONES_DIR.mkdir(exist_ok=True)

motif = resolve_target_motif(TARGET_PDB, target_chain='B', motif_residues=MOTIF_RESIDUES)
spec = MotifSpec(
    target_motif=motif,
    binder_length_min=BINDER_LEN_MIN,
    binder_length_max=BINDER_LEN_MAX,
)

rng = random.Random(42)
for i in range(NUM_BACKBONES):
    seed = rng.randrange(1, 2**31 - 1)
    contig = build_contig_string(spec, rng=random.Random(seed))
    out_prefix = BACKBONES_DIR / f'design_{i:04d}'
    print(f'[{i+1}/{NUM_BACKBONES}] contig={contig}')
    !cd /content/RFdiffusion && python scripts/run_inference.py \
        inference.input_pdb={TARGET_PDB} \
        "contigmap.contigs=[{contig}]" \
        inference.output_prefix={out_prefix} \
        inference.num_designs=1 \
        diffuser.T={DIFFUSER_T} \
        inference.deterministic=True \
        inference.random_seed={seed} 2>&1 | tail -3

ModuleNotFoundError: No module named 'binderdiffuser'

In [ ]:
# Sanity check: how many backbone PDBs landed?
backbone_pdbs = sorted(BACKBONES_DIR.glob('design_*.pdb'))
print(f'generated {len(backbone_pdbs)} backbones')
for p in backbone_pdbs[:3]:
    print(' ', p.name)

## 5. Design sequences with ProteinMPNN

For each backbone, generate `SEQS_PER_BACKBONE` candidate sequences. The
target chain residues are held fixed (we are not redesigning the target).

In [ ]:
import json

MPNN_DIR = OUTPUT_DIR / 'sequences'
MPNN_DIR.mkdir(exist_ok=True)

for bb_pdb in backbone_pdbs:
    design_id = bb_pdb.stem
    seq_out = MPNN_DIR / design_id
    seq_out.mkdir(exist_ok=True)
    chain_jsonl = MPNN_DIR / f'{design_id}_chains.jsonl'
    # RFdiffusion's binder convention: binder = chain A, target = chain B.
    chain_jsonl.write_text(json.dumps({design_id: [['A'], ['B']]}) + '\n')
    pdb_dir = seq_out / 'pdb_input'
    pdb_dir.mkdir(exist_ok=True)
    !cp {bb_pdb} {pdb_dir}/
    !python /content/ProteinMPNN/protein_mpnn_run.py \
        --pdb_path {pdb_dir} \
        --pdb_path_chains {chain_jsonl} \
        --out_folder {seq_out} \
        --num_seq_per_target {SEQS_PER_BACKBONE} \
        --sampling_temp 0.1 \
        --seed 37 \
        --batch_size 1 2>&1 | tail -1

In [ ]:
# Aggregate all MPNN-designed sequences into one FASTA for ColabFold.
from binderdiffuser.sequence.mpnn_runner import ProteinMPNNRunner

mpnn = ProteinMPNNRunner(check_executable=False)
all_seqs = []
for bb_pdb in backbone_pdbs:
    design_id = bb_pdb.stem
    fa_files = list((MPNN_DIR / design_id / 'seqs').glob('*.fa'))
    for fa in fa_files:
        all_seqs.extend(mpnn.parse_output_fasta(fa, backbone_id=design_id))

# For multimer folding we concatenate binder + ':' + target sequence.
TARGET_SEQ = ''  # populate below from the trimmed PDB
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import three_to_one
structure = PDBParser(QUIET=True).get_structure('t', TARGET_PDB)
for residue in next(structure.get_models())['B']:
    if residue.id[0] == ' ':
        try:
            TARGET_SEQ += three_to_one(residue.resname)
        except KeyError:
            TARGET_SEQ += 'X'
print('target length:', len(TARGET_SEQ))

fasta_path = OUTPUT_DIR / 'all_designs.fasta'
lines = []
for s in all_seqs:
    seq_id = f'{s.backbone_id}_seq{s.sequence_index}'
    lines.append(f'>{seq_id}\n{s.sequence}:{TARGET_SEQ}\n')
fasta_path.write_text(''.join(lines))
print(f'wrote {len(all_seqs)} multimer sequences to {fasta_path}')

## 6. Re-fold each design with ColabFold (AF2 multimer)

This is the slowest step. With `num_recycle=3` and `num_models=1`, expect
~20 seconds per sequence on T4.

We use `single_sequence` MSA mode because the binder is de novo — there
are no homologs.

In [ ]:
AF_DIR = OUTPUT_DIR / 'alphafold'
AF_DIR.mkdir(exist_ok=True)
!colabfold_batch \
  --msa-mode single_sequence \
  --num-recycle 3 \
  --num-models 1 \
  --rank multimer \
  --model-type alphafold2_multimer_v3 \
  {fasta_path} {AF_DIR}

## 7. Compute metrics, filter, rank

Now we reuse the BinderDiffuser metric and filter modules to produce the
final ranked CSV.

In [ ]:
import json
import numpy as np
import pandas as pd
from binderdiffuser.config import FilterConfig
from binderdiffuser.validation.filters import (
    DesignRecord, composite_score, filter_designs, rank_designs, to_dataframe,
)
from binderdiffuser.validation.metrics import (
    compute_iptm, compute_pae_interface, compute_plddt_summary,
    compute_sc_rmsd, compute_sc_tm,
)

records: list[DesignRecord] = []
for af_pdb in sorted(AF_DIR.glob('*_unrelaxed_rank_001_*.pdb')):
    seq_id = af_pdb.name.split('_unrelaxed_rank_')[0]
    backbone_id, _, idx = seq_id.rpartition('_seq')
    seq_idx = int(idx)
    bb_pdb = BACKBONES_DIR / f'{backbone_id}.pdb'
    score_json = next(AF_DIR.glob(f'{seq_id}_scores_rank_001_*.json'), None)
    if score_json is None or not bb_pdb.exists():
        continue
    with score_json.open() as f:
        scores = json.load(f)

    seq = next((s for s in all_seqs if s.backbone_id == backbone_id and s.sequence_index == seq_idx), None)
    if seq is None:
        continue
    binder_len = len(seq.sequence)
    target_len = len(TARGET_SEQ)

    sc_rmsd = compute_sc_rmsd(str(bb_pdb), str(af_pdb), binder_chain='A')
    sc_tm = compute_sc_tm(str(bb_pdb), str(af_pdb), binder_chain='A')
    plddt = scores.get('plddt', [])
    overall, binder_plddt = compute_plddt_summary(plddt, binder_length=binder_len)
    iptm = compute_iptm(scores)
    pae_iface = None
    if 'pae' in scores:
        try:
            pae_iface = compute_pae_interface(scores['pae'], binder_len, target_len)
        except ValueError as e:
            print('pae skipped', seq_id, e)

    records.append(DesignRecord(
        design_id=seq_id, backbone_id=backbone_id, sequence_index=seq_idx,
        sequence=seq.sequence, sc_rmsd=sc_rmsd, sc_tm=sc_tm,
        mean_plddt=overall, mean_plddt_binder=binder_plddt,
        iptm=iptm, pae_interface=pae_iface,
        binder_length=binder_len, target_length=target_len,
        mpnn_score=seq.score,
    ))

print(f'computed metrics for {len(records)} designs')
df = to_dataframe(records)
csv_path = OUTPUT_DIR / 'designs.csv'
df.to_csv(csv_path, index=False)
df.head(10)

In [ ]:
# Apply filters and show the ranked top 5.
filt = filter_designs(records, FilterConfig())
ranked = rank_designs(filt, top_k=5)
print(f'kept {len(filt)}/{len(records)} after filtering, top {len(ranked)} ranked\n')
for r in ranked:
    print(f'{r.design_id}: scRMSD={r.sc_rmsd:.2f}  scTM={r.sc_tm:.2f}  '
          f'pLDDT={r.mean_plddt:.1f}  ipTM={r.iptm:.2f}  '
          f'pAE_iface={r.pae_interface:.2f}  composite={composite_score(r):.2f}')

## 8. Generate the README plots from this real run

In [ ]:
from binderdiffuser.viz import metrics_violin, self_consistency_scatter

fig_dir = OUTPUT_DIR / 'figures'
fig_dir.mkdir(exist_ok=True)
self_consistency_scatter(df, fig_dir / 'sc_scatter.png',
                          title=f'PD-L1 binder cohort (n={len(df)}, real run)')
metrics_violin(df, fig_dir / 'metrics_violin.png')
print('figures written to', fig_dir)

from IPython.display import Image, display
display(Image(str(fig_dir / 'sc_scatter.png')))
display(Image(str(fig_dir / 'metrics_violin.png')))

## 9. Download artifacts

Tarball the run and download to your local machine. You can then commit
`figures/` and `designs.csv` back to the repo to replace the illustrative
cohort with real numbers.

In [ ]:
import shutil
tarball = f'/content/{RUN_NAME}.tar.gz'
!tar czf {tarball} -C {OUTPUT_DIR.parent} {RUN_NAME}
print('size:', Path(tarball).stat().st_size / 1e6, 'MB')
from google.colab import files
files.download(tarball)

## What to do with the results

1. **Inspect `designs.csv`** — these are real metrics on a real cohort.
2. **Replace `figures/sample_designs.csv`** in the repo with this run's CSV
   to make the README plots reflect actual data.
3. **In interviews**, you can now point to specific numbers (e.g. "10
   backbones, 40 sequences, 8 passed filters with scRMSD < 2 Å and
   pLDDT > 70").
4. **Commit the run summary** but NOT the raw PDBs (too large) — keep them
   in Drive or a Releases asset.

## Bumping up the run

Once the smoke test works, increase `NUM_BACKBONES` to 50-100 and
`SEQS_PER_BACKBONE` to 8 for a publication-grade cohort. On A100 expect
~2 hours; on T4, ~6-8 hours (use Colab Pro for longer sessions).